# Create Unity Catalog Functions for Policies and Customer Service Tables
Creates reusable SQL functions for querying and analyzing policies and customer service data.

**Source Tables:**
- `llmagent.dev.policies` - Policy information (policy, policy_details, last_updated)
- `llmagent.dev.cust_service_data` - Customer service interactions

**Functions Created:**
1. `get_customer_interactions()` - Get all customer interactions
2. `search_customer_by_email()` - Search customers by email
3. `get_issues_by_category()` - Get issues filtered by category
4. `get_customer_service_summary()` - Get customer service statistics

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("CreateUCFunctions").getOrCreate()

# Configuration
CATALOG = "llmagent"
SCHEMA = "dev"
POLICIES_TABLE = f"{CATALOG}.{SCHEMA}.policies"
CUST_SERVICE_TABLE = f"{CATALOG}.{SCHEMA}.cust_service_data"

print(f"Catalog: {CATALOG}")
print(f"Schema: {SCHEMA}")
print(f"Policies Table: {POLICIES_TABLE}")
print(f"Customer Service Table: {CUST_SERVICE_TABLE}")

## Function 1: Get Customer Interactions
Returns customer service interaction details with customer information.

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.get_customer_interactions()
    RETURNS TABLE(
        customer_id STRING,
        customer_name STRING,
        email STRING,
        phone STRING,
        interaction_id STRING,
        date_time TIMESTAMP,
        issue_category STRING,
        issue_description STRING,
        agent_id BIGINT
    )
    LANGUAGE SQL
    AS
    SELECT
        customer_id,
        name as customer_name,
        email,
        phone_number as phone,
        interaction_id,
        date_time,
        issue_category,
        issue_description,
        agent_id
    FROM {CUST_SERVICE_TABLE}
    ORDER BY date_time DESC
""")

print("✓ Function created: get_customer_interactions()")

## Function 2: Search Customer by Email
Search for customers and their service interactions by email address.

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.search_customer_by_email(email_input STRING)
    RETURNS TABLE(
        customer_id STRING,
        customer_name STRING,
        email STRING,
        phone STRING,
        address STRING,
        interaction_count BIGINT,
        last_interaction TIMESTAMP
    )
    LANGUAGE SQL
    AS
    SELECT
        customer_id,
        name as customer_name,
        email,
        phone_number as phone,
        address,
        COUNT(interaction_id) as interaction_count,
        MAX(date_time) as last_interaction
    FROM {CUST_SERVICE_TABLE}
    WHERE email = email_input
    GROUP BY customer_id, name, email, phone_number, address
""")

print("✓ Function created: search_customer_by_email(email_input)")

## Function 3: Get Issues by Category
Retrieve all customer service issues filtered by category.

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.get_issues_by_category(category_input STRING)
    RETURNS TABLE(
        customer_id STRING,
        customer_name STRING,
        interaction_id STRING,
        date_time TIMESTAMP,
        issue_category STRING,
        issue_description STRING,
        agent_id BIGINT
    )
    LANGUAGE SQL
    AS
    SELECT
        customer_id,
        name as customer_name,
        interaction_id,
        date_time,
        issue_category,
        issue_description,
        agent_id
    FROM {CUST_SERVICE_TABLE}
    WHERE issue_category = category_input
    ORDER BY date_time DESC
""")

print("✓ Function created: get_issues_by_category(category_input)")

## Function 4: Get Customer Service Summary
Get summary statistics for customer service interactions.

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.get_customer_service_summary()
    RETURNS TABLE(
        total_customers BIGINT,
        total_interactions BIGINT,
        unique_issues BIGINT,
        active_agents BIGINT,
        avg_interactions_per_customer DOUBLE,
        earliest_interaction TIMESTAMP,
        latest_interaction TIMESTAMP
    )
    LANGUAGE SQL
    AS
    SELECT
        COUNT(DISTINCT customer_id) as total_customers,
        COUNT(interaction_id) as total_interactions,
        COUNT(DISTINCT issue_category) as unique_issues,
        COUNT(DISTINCT agent_id) as active_agents,
        ROUND(COUNT(interaction_id) / COUNT(DISTINCT customer_id), 2) as avg_interactions_per_customer,
        MIN(date_time) as earliest_interaction,
        MAX(date_time) as latest_interaction
    FROM {CUST_SERVICE_TABLE}
""")

print("✓ Function created: get_customer_service_summary()")

## Function 5: Get Customer Issues by Date Range
Retrieve customer service issues within a specific date range.

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.get_issues_by_date_range(
        start_date TIMESTAMP,
        end_date TIMESTAMP
    )
    RETURNS TABLE(
        customer_id STRING,
        customer_name STRING,
        interaction_id STRING,
        date_time TIMESTAMP,
        issue_category STRING,
        issue_description STRING,
        agent_id BIGINT
    )
    LANGUAGE SQL
    AS
    SELECT
        customer_id,
        name as customer_name,
        interaction_id,
        date_time,
        issue_category,
        issue_description,
        agent_id
    FROM {CUST_SERVICE_TABLE}
    WHERE date_time BETWEEN start_date AND end_date
    ORDER BY date_time DESC
""")

print("✓ Function created: get_issues_by_date_range(start_date, end_date)")

## Function 6: Get Agent Performance
Get performance statistics for each support agent.

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.get_agent_performance()
    RETURNS TABLE(
        agent_id BIGINT,
        total_interactions BIGINT,
        unique_customers BIGINT,
        unique_issue_categories BIGINT,
        earliest_interaction TIMESTAMP,
        latest_interaction TIMESTAMP
    )
    LANGUAGE SQL
    AS
    SELECT
        agent_id,
        COUNT(interaction_id) as total_interactions,
        COUNT(DISTINCT customer_id) as unique_customers,
        COUNT(DISTINCT issue_category) as unique_issue_categories,
        MIN(date_time) as earliest_interaction,
        MAX(date_time) as latest_interaction
    FROM {CUST_SERVICE_TABLE}
    GROUP BY agent_id
    ORDER BY total_interactions DESC
""")

print("✓ Function created: get_agent_performance()")

## List All Created Functions

In [ ]:
# Show all functions in the schema
display(spark.sql(f"""
    SHOW FUNCTIONS IN {CATALOG}.{SCHEMA} LIKE 'get_*'
"""))

## Test Functions

In [ ]:
# Test: Get customer service summary
print("Test 1: Customer Service Summary")
display(spark.sql(f"""
    SELECT * FROM {CATALOG}.{SCHEMA}.get_customer_service_summary()
"""))

In [ ]:
# Test: Get agent performance
print("Test 2: Agent Performance Statistics")
display(spark.sql(f"""
    SELECT * FROM {CATALOG}.{SCHEMA}.get_agent_performance() LIMIT 10
"""))

In [ ]:
# Test: Get customer interactions (first 5)
print("Test 3: Recent Customer Interactions")
display(spark.sql(f"""
    SELECT * FROM {CATALOG}.{SCHEMA}.get_customer_interactions() LIMIT 5
"""))

In [ ]:
# Test: Get issues by category
print("Test 4: Issues by Category (first issue category found)")
spark.sql(f"""
    SELECT DISTINCT issue_category FROM {CUST_SERVICE_TABLE} LIMIT 1
""").collect()

category = spark.sql(f"SELECT DISTINCT issue_category FROM {CUST_SERVICE_TABLE} LIMIT 1").collect()[0][0]
print(f"Using category: {category}")

display(spark.sql(f"""
    SELECT * FROM {CATALOG}.{SCHEMA}.get_issues_by_category('{category}') LIMIT 5
"""))

## Summary of Created Functions

All functions are now available in the Unity Catalog and can be called as:

1. **get_customer_interactions()** - Returns all customer interactions
2. **search_customer_by_email(email_input)** - Search customers by email
3. **get_issues_by_category(category_input)** - Get issues by category
4. **get_customer_service_summary()** - Summary statistics
5. **get_issues_by_date_range(start_date, end_date)** - Issues within date range
6. **get_agent_performance()** - Agent performance metrics

### Usage Examples:
```sql
-- Get summary
SELECT * FROM llmagent.dev.get_customer_service_summary();

-- Search customer by email
SELECT * FROM llmagent.dev.search_customer_by_email('customer@email.com');

-- Get agent performance
SELECT * FROM llmagent.dev.get_agent_performance();
```